## 2. Chart: Exploring the data and creating visualisation
### 2.1 Load the clean data

In [5]:
import pandas as pd

clean_data = pd.read_csv("data/clean_data.csv", parse_dates=["month"], dtype={"coicop_code": str})
clean_data.head()

,month,geo,coicop_code,category,value
0,2019-01-01,Austria,00,All items,1.7
1,2019-02-01,Austria,00,All items,1.4
2,2019-03-01,Austria,00,All items,1.7
3,2019-04-01,Austria,00,All items,1.7
4,2019-05-01,Austria,00,All items,1.7


In [6]:
clean_data.shape

(9408, 5)

### 2.2 Exploratory chart 1 - Ireland vs Euro (all items)

In [7]:
# Source: Plotly line charts, https://plotly.com/python/line-charts/
import plotly.express as px

# Keep only "All items" for Ireland and the Euro area
ireland_vs_euro = clean_data[(clean_data["category"] == "All items") & 
                        (clean_data["geo"].isin(["Ireland", "Euro area"]))]

chart1 = px.line(ireland_vs_euro, x="month", y="value", color="geo",
                title="Exploratory: Ireland vs Euro area, all items")
chart1.show()

What: A simple line chart of Ireland and the Euro area for all the items, Jan 2019 - Dec 2025. 

Why: To see the overall story before designing the final chart.
This is an exploratory chart, not the final visualisation.

Result: Both lines rise sharply in 2022. Ireland peaked at 9.6% (in June 2022), a little lower than the Euro area at 10.6% (in October 2022). So overall, prices in Ireland did not rise faster than in the Euro area.

### 2.3 Exploratory chart 2 - Ireland vs Euro area by category

In [8]:
# Source: pandas group by, https://pandas.pydata.org/docs/user_guide/groupby.html
# and plotly horizontal bar charts, https://plotly.com/python/horizontal-bar-charts/

# keep only the crisis years, 2022 and 2023
crisis = clean_data[clean_data["month"].dt.year.isin([2022, 2023])]

# Average rate for each category and place
average = crisis.groupby(["category", "geo"])["value"].mean().unstack()

# Ireland minus Euro area: above 0 means prices rose faster in Ireland
gap = (average["Ireland"] - average["Euro area"]).reset_index(name="gap")
gap = gap.sort_values("gap")

chart2 = px.bar(gap, x="gap", y="category", orientation="h",
                title="Exploratory: Ireland minus Euro area, 2022-2023")
chart2.show()

What: The average gap between Ireland and the Euro area for each category in 2022-2023

Why: To see which category rose faster in Ireland. This decides what the final chart needs to show.
This is an exploratory chart, not the final visualisation.

Result: Housing (+8.1) and Energy (+6.0) rose faster in Ireland than in Euro area. Food (-2.8) rose slower, and All items (-0.3) was almost the same. So the final chart should focus on Housing and Energy.

### 2.4 Band data - middle half of EU countries

In [9]:
# Keep only the 27 EU countries (not the Euro area)
countries = clean_data[clean_data["geo"] != "Euro area"]

# For each category and month: bottom and top of the middle half of countries
low = countries.groupby(["category", "month"])["value"].quantile(0.25)
high = countries.groupby(["category", "month"])["value"].quantile(0.75)

band = pd.DataFrame({"low": low, "high": high}).reset_index()
band.head()

,category,month,low,high
0,All items,2019-01-01,1.00,2.05
1,All items,2019-02-01,1.10,2.20
2,All items,2019-03-01,1.15,2.60
3,All items,2019-04-01,1.50,2.55
4,All items,2019-05-01,1.00,2.55


What: For each category and month, the range of the middle half of EU countries.

Why: A grey band is cleaner than 27 grey lines. A few extreme countries (up to +114%) stretched the chart and hid the Ireland gap.

Result: 336 rows (4 categories × 84 months), with columns low and high.

### 2.5 Final chart - one category first (Housing)

In [10]:
# Source: Plotly continuous error bands, https://plotly.com/python/continuous-error-bars/
# and Plotly line charts, https://plotly.com/python/line-charts/

import plotly.graph_objects as go

# Test with one category first
category = "Housing"

# Data for this category
band_one = band[band["category"] == category]
ireland_line = clean_data[(clean_data["category"] == category) & (clean_data["geo"] == "Ireland")]
euro_line = clean_data[(clean_data["category"] == category) & (clean_data["geo"] == "Euro area")]

final_chart = go.Figure()

# Grey band: bottom edge (no line), then top edge filled down to it
final_chart.add_trace(go.Scatter(x=band_one["month"], y=band_one["low"],
                                 line=dict(width=0), showlegend=False))
final_chart.add_trace(go.Scatter(x=band_one["month"], y=band_one["high"],
                                 line=dict(width=0), fill="tonexty",
                                 fillcolor="lightgrey", name="Middle half of EU countries"))

# Euro area: dashed dark blue line
final_chart.add_trace(go.Scatter(x=euro_line["month"], y=euro_line["value"], name="Euro area",
                                 line=dict(color="#1F3B73", width=2, dash="dash")))

# Ireland: thick green line
final_chart.add_trace(go.Scatter(x=ireland_line["month"], y=ireland_line["value"], name="Ireland",
                                 line=dict(color="#00843D", width=4)))

final_chart.update_layout(title="Housing: Ireland vs the rest of Europe",
                          yaxis_title="Annual price change (%)",
                          template="simple_white")
final_chart.show()

What: First version of the final chart for one category (Housing).

Why: To test the band and the two lines before adding all 4 categories.

Result: Ireland's green line is above the grey band in 2022–2023, so housing costs rose faster in Ireland than in most EU countries.

### 2.6 Final chart - all 4 categories with button

In [11]:
# Source: Plotly custom buttons, https://plotly.com/python/custom-buttons/
# and Plotly continuous error bands, https://plotly.com/python/continuous-error-bars/

categories = ["All items", "Food", "Housing", "Energy"]

# Headline title for each button (numbers from Step 2.3)
titles = {
    "All items": "Overall, Irish prices rose about as fast as the Euro area in 2022–23",
    "Food": "Irish food prices rose the slowest in the EU in 2022–23",
    "Housing": "Irish housing costs rose almost twice as fast as the Euro area in 2022–23",
    "Energy": "Irish energy prices rose faster than the Euro area in 2022–23",
}

final_chart = go.Figure()

# Same 4 lines as Step 2.6, once for each category
for category in categories:
    band_one = band[band["category"] == category]
    ireland_line = clean_data[(clean_data["category"] == category) & (clean_data["geo"] == "Ireland")]
    euro_line = clean_data[(clean_data["category"] == category) & (clean_data["geo"] == "Euro area")]

    # Only "All items" is shown when the chart opens
    show = (category == "All items")

    final_chart.add_trace(go.Scatter(x=band_one["month"], y=band_one["low"],
                                     line=dict(width=0), showlegend=False, visible=show))
    final_chart.add_trace(go.Scatter(x=band_one["month"], y=band_one["high"],
                                     line=dict(width=0), fill="tonexty", fillcolor="lightgrey",
                                     name="Middle half of EU countries", visible=show))
    final_chart.add_trace(go.Scatter(x=euro_line["month"], y=euro_line["value"], name="Euro area",
                                     line=dict(color="#1F3B73", width=2, dash="dash"), visible=show))
    final_chart.add_trace(go.Scatter(x=ireland_line["month"], y=ireland_line["value"], name="Ireland",
                                     line=dict(color="#00843D", width=4), visible=show))

# One button per category
buttons = []
for i, category in enumerate(categories):
    # 16 lines in total (4 per category). Switch on only this category's 4 lines
    visible = [False] * 16
    visible[i * 4 : i * 4 + 4] = [True] * 4

    buttons.append(dict(label=category, method="update",
                        args=[{"visible": visible},
                              {"title.text": titles[category], "yaxis.autorange": True}]))

final_chart.update_layout(title=titles["All items"],
                          yaxis_title="Annual price change (%)",
                          template="simple_white",
                          updatemenus=[dict(type="buttons", direction="right", buttons=buttons,
                                            x=0, xanchor="left", y=1.15)])
final_chart.show()

What: The final chart with all 4 categories and a button for each one.

Why: Lets the user switch between categories to see which ones drove the crisis. The title changes to state what each view shows.

Result: All items opens first. Clicking Food, Housing or Energy redraws the same chart with that category's data and title.